In [8]:
import networkx as nx
import numpy as np
import collections

def construct_Gn(n):
    """
    Constructs the graph G_n: a complete graph K_n with two pendant
    vertices attached to two different vertices of K_n.
    
    Here, we attach them to nodes 0 and 1.
    """
    if n < 2:
        raise ValueError("n must be at least 2 for G_n to be well-defined.")
        
    # 1. Create the complete graph K_n on nodes 0 to n-1
    G = nx.complete_graph(n)
    
    # 2. Add the two new pendant vertices
    pendant_node_1 = n
    pendant_node_2 = n + 1
    
    G.add_nodes_from([pendant_node_1, pendant_node_2])
    
    # 3. Attach them to two different vertices of K_n (e.g., node 0 and node 1)
    G.add_edge(0, pendant_node_1)
    G.add_edge(1, pendant_node_2)
    
    return G

def get_spectrum(G):
    """
    Calculates the spectrum (eigenvalues of the adjacency matrix)
    of a graph G.
    
    Returns a sorted tuple of eigenvalues rounded to 8 decimal
    places to allow for consistent comparison.
    """
    # Get the adjacency matrix as a dense numpy array
    A = nx.adjacency_matrix(G).toarray()
    
    # Calculate eigenvalues
    # eigvalsh is used for symmetric matrices (like adjacency matrices)
    eigenvalues = np.linalg.eigvalsh(A)
    
    # Round to handle floating-point inaccuracies
    rounded_eigenvalues = np.round(eigenvalues, 8)
    
    # Sort for consistent comparison
    rounded_eigenvalues.sort()
    
    # Return as a tuple to make it hashable
    return tuple(rounded_eigenvalues.tolist())

def check_das_for_small_n(n_values_to_check):
    """
    Checks if G_n is Determined by its Adjacency Spectrum (DAS)
    for a list of small n values.
    
    It does this by:
    1. Constructing G_n and finding its spectrum.
    2. Generating ALL non-isomorphic graphs 'H' with the same
       number of vertices (n+2).
    3. Finding the spectrum of each H.
    4. Counting how many graphs 'H' share the same spectrum as G_n.
    5. If the count is > 1, it means non-isomorphic cospectral
       graphs exist, and G_n is NOT DAS.
    """
    
    print("--- Starting Cospectral Check ---")
    
    # Get the graph atlas (all non-isomorphic graphs up to 7 vertices)
    # This is necessary because iterating through all graphs for n=6 (8 vertices)
    # is computationally very expensive (12,001 graphs).
    # n=3 -> 5 vertices (34 graphs)
    # n=4 -> 6 vertices (156 graphs)
    # n=5 -> 7 vertices (1,044 graphs)
    try:
        all_graphs_atlas = list(nx.graph_atlas_g())
    except ImportError:
        print("Error: Could not load graph_atlas_g().")
        print("Please ensure networkx is correctly installed.")
        return

    for n in n_values_to_check:
        num_vertices = n + 2
        
        if num_vertices > 7:
            print(f"\nSkipping n={n} (order {num_vertices}): "
                  f"Checking all graphs of order > 7 is too slow "
                  f"for this script.")
            continue

        print(f"\n--- Checking for n = {n} (Graph order {num_vertices}) ---")
        
        # 1. Get the target graph G_n and its spectrum
        try:
            Gn = construct_Gn(n)
            spec_Gn = get_spectrum(Gn)
            print(f"Spectrum of G_{n}: {spec_Gn}")
        except ValueError as e:
            print(f"Could not construct G_{n}: {e}")
            continue

        # 2. Filter atlas for graphs with the correct number of vertices
        graphs_to_check = [
            H for H in all_graphs_atlas 
            if H.number_of_nodes() == num_vertices
        ]
        
        print(f"Checking against {len(graphs_to_check)} non-isomorphic graphs...")

        # 3. Store spectra in a dictionary to count occurrences
        spectra_count = collections.defaultdict(list)
        
        Gn_isomorphic_found = False
        
        for H in graphs_to_check:
            spec_H = get_spectrum(H)
            
            # Check if this graph H is the one we are looking for
            if spec_H == spec_Gn:
                # We found a graph with the same spectrum.
                # Store it. We'll check for isomorphism later.
                spectra_count[spec_H].append(H)
                
                if not Gn_isomorphic_found:
                    if nx.is_isomorphic(Gn, H):
                        Gn_isomorphic_found = True


        # 4. Analyze the results
        cospectral_graphs = spectra_count.get(spec_Gn, [])
        num_cospectral = len(cospectral_graphs)

        if not Gn_isomorphic_found:
            # This should not happen if the atlas is correct
            print(f"Error: G_{n} was not found in the graph atlas.")
        elif num_cospectral == 1:
            # Only one graph (the one isomorphic to G_n) has this spectrum
            print(f"RESULT: G_{n} is DAS for n = {n}.")
            print("No other non-isomorphic graphs share its spectrum.")
        else:
            # More than one graph has the same spectrum
            print(f"RESULT: G_{n} is NOT DAS for n = {n}.")
            print(f"Found {num_cospectral} total graphs "
                  f"(including G_{n}) with the same spectrum.")
            
            # Optional: List the non-isomorphic ones
            print("Non-isomorphic cospectral graphs found:")
            for H in cospectral_graphs:
                if not nx.is_isomorphic(Gn, H):
                    print(f"  - Graph with edges: {list(H.edges())}")

# --- Main execution ---
if __name__ == "__main__":
    # We can check for n=3, 4, 5, as they result in graphs of
    # order 5, 6, and 7, which are in the standard NetworkX atlas.
    n_to_check = [3, 4, 5]
    check_das_for_small_n(n_to_check)

--- Starting Cospectral Check ---

--- Checking for n = 3 (Graph order 5) ---
Spectrum of G_3: (-1.61803399, -1.30277564, -0.0, 0.61803399, 2.30277564)
Checking against 34 non-isomorphic graphs...
RESULT: G_3 is DAS for n = 3.
No other non-isomorphic graphs share its spectrum.

--- Checking for n = 4 (Graph order 6) ---
Spectrum of G_4: (-1.61803399, -1.39138238, -1.0, 0.22713444, 0.61803399, 3.16424794)
Checking against 156 non-isomorphic graphs...
RESULT: G_4 is DAS for n = 4.
No other non-isomorphic graphs share its spectrum.

--- Checking for n = 5 (Graph order 7) ---
Spectrum of G_5: (-1.61803399, -1.43931167, -1.0, -1.0, 0.33887969, 0.61803399, 4.10043199)
Checking against 1044 non-isomorphic graphs...
RESULT: G_5 is DAS for n = 5.
No other non-isomorphic graphs share its spectrum.


In [9]:
check_das_for_small_n(range(1,8))

--- Starting Cospectral Check ---

--- Checking for n = 1 (Graph order 3) ---
Could not construct G_1: n must be at least 2 for G_n to be well-defined.

--- Checking for n = 2 (Graph order 4) ---
Spectrum of G_2: (-1.61803399, -0.61803399, 0.61803399, 1.61803399)
Checking against 11 non-isomorphic graphs...
RESULT: G_2 is DAS for n = 2.
No other non-isomorphic graphs share its spectrum.

--- Checking for n = 3 (Graph order 5) ---
Spectrum of G_3: (-1.61803399, -1.30277564, -0.0, 0.61803399, 2.30277564)
Checking against 34 non-isomorphic graphs...
RESULT: G_3 is DAS for n = 3.
No other non-isomorphic graphs share its spectrum.

--- Checking for n = 4 (Graph order 6) ---
Spectrum of G_4: (-1.61803399, -1.39138238, -1.0, 0.22713444, 0.61803399, 3.16424794)
Checking against 156 non-isomorphic graphs...
RESULT: G_4 is DAS for n = 4.
No other non-isomorphic graphs share its spectrum.

--- Checking for n = 5 (Graph order 7) ---
Spectrum of G_5: (-1.61803399, -1.43931167, -1.0, -1.0, 0.3388796